# Final report on data challenge "Maintenance and Industry 4.0 2026"

## Group 7 - Members

- Pedro de Bem
- Efe Olgun
- Aziz Hamza

## Submission
- Notebook submitted via the course portal (https://nextcloud.centralesupelec.fr/s/bcGcQzimE4ZjetS)
- github repository [https://github.com/debem-cs/robot-predictive-maintenance]
- Final Kaggle result: 1st place, private score 0.86323 (public 0.76204).


## Methods

### Problem and key observation
For each of the 6 servo motors and each timestep of each test sequence, we predict
whether an abnormal temperature rise is present. Submissions are scored by the mean
F1 across the six motors. The fault is injected synthetically: a non-linear additive
thermal pulse is added to the temperature channel over a contiguous span, while the
voltage and position channels stay normal. A fault is therefore a positive, sustained
increase in temperature relative to what the motor would otherwise read.

### Strategy: unsupervised per-sequence residual detection
Our first attempts (a SMOTE + RandomForest classifier, then a learned
temperature-regression digital twin with absolute residuals) failed on the blind
test set. The test faults are subtler than the few training faults, so fixed learned
thresholds either never fired (close to zero faults predicted) or over-flagged. We
moved to a method that does not depend on a model generalising across the train/test
gap.

For each motor and each sequence independently:

```
expected(t) = centred rolling-median of temperature   (within-sequence baseline)
residual(t) = temperature(t) - expected(t)
```

The rolling median follows slow legitimate warming but is barely moved by a short
fault pulse, so the residual isolates the injected anomaly. We normalise the residual
per sequence by its median and MAD (a robust z-score), which makes the detector
scale-invariant and transferable across the train/test gap. Detection uses hysteresis
to use the fault's asymmetric-triangle shape (fast heat-up, slower cool-down):

* a point seeds a fault when its residual is above k_high robust sigma and above
  min_rise degrees of the sequence's normal level (an absolute floor that removes
  1-LSB temperature-quantisation noise);
* the seed grows over neighbours that clear a gentler k_low threshold, capturing the
  triangle ramp tails that a single cutoff would clip;
* run-length rules keep only sustained runs (min_run) and bridge short holes (gap).
  Detection is signed, so only temperature rises are flagged.

### Class imbalance
Imbalance is the main difficulty. Of the original 23 training sequences only 7 contain
any fault, and after discarding one degenerate sequence (20240325_155003, motors 2 and
4 flagged for all rows, with no usable baseline), motors 1 to 5 each have a fault in
only one sequence. We handled this in three ways:

1. Shared calibration for motors 1 to 5. The detection knobs are tuned jointly across
   motors 1 to 5, pooling their few positives, with motor 6 calibrated separately.
   Selection maximises macro F1 (the mean of the per-motor F1), matching the competition
   metric so no subtle motor is dropped.
2. Synthetic augmentation. We ported the competition's failure-injection routine to
   Python and injected extra labelled faults into the fault-free sequences. The whole
   dataset, including the hidden test labels, is produced by that same generator, so
   synthetic faults follow the test distribution. A held-out synthetic score then
   became a usable proxy for the leaderboard, which the single subtle real fault
   sequence per motor could not provide.
3. Additional labelled data. Extra labelled sets released later (groups 1, 6 and 7)
   gave several real fault sequences per motor and a real held-out validation set. They
   confirmed that a larger rolling-baseline window generalises better (window 400, then
   500, then 600), which the leaderboard then confirmed.

### Final model
The same detector is used for every motor; only the rolling window and the
shared-versus-separate calibration differ. The feature is the temperature residual;
voltage and position are used to establish that the fault is temperature-specific, not
as model inputs.

| Motors | Final model | Features | Dataset used for calibration | Performance (synthetic generator-matched F1) | Additional notes |
| --- | --- | --- | --- | --- | --- |
| Motor 1 | Per-sequence residual + hysteresis (W=600) | Temperature residual vs rolling-median baseline | Original + additional training data + synthetic faults | 0.85 | shared knobs (motors 1 to 5) |
| Motor 2 | same (W=600) | same | same | 0.93 | shared knobs |
| Motor 3 | same (W=600) | same | same | 0.85 | subtle ~1 C ramps; hysteresis recovers them |
| Motor 4 | same (W=600) | same | same | 0.87 | shared knobs |
| Motor 5 | same (W=600) | same | same | 0.87 | shared knobs |
| Motor 6 | same (W=300) | same | same | 0.75 | gripper; separate knobs, hardest motor |

Common knobs: k_high=4, k_low=1, min_rise=0.5 C, min_run=3, gap=3.


## Results

The detector and calibration code live in the project package
(robot-predictive-maintenance/src/): preprocessing.py (robust cleaning), detector.py
(the residual and hysteresis detector, the core), augmentation.py (the synthetic fault
generator), and run_pipeline.py (calibration and submission). Below we load the data,
apply the final calibrated knobs, report the per-motor performance, and regenerate the
winning submission. The full grid calibration is reproducible with
`python src/run_pipeline.py`; here we apply its result so the notebook runs quickly.

In [1]:
import sys, os
import numpy as np, pandas as pd

SRC = os.path.join('src')
sys.path.insert(0, SRC)
import preprocessing as pp          # robust cleaning
import detector as det              # residual + hysteresis detector (core)
import augmentation as aug          # synthetic fault generator
import run_pipeline as rp           # calibration + submission pipeline
import sweep as sw                  # offline evaluation helpers

# All labelled data: the original challenge training set + the additional sets.
df_train = rp.load(rp.TRAIN_DIR)
df_extra = rp.load_tree(rp.EXTRA_DIR)
df_all   = pd.concat([df_train, df_extra]).reset_index(drop=True)
print(f'original training : {df_train.test_condition.nunique():2d} sequences, {len(df_train):6d} rows')
print(f'additional data   : {df_extra.test_condition.nunique():2d} sequences, {len(df_extra):6d} rows')
print(f'combined          : {df_all.test_condition.nunique():2d} sequences, {len(df_all):6d} rows')

original training : 23 sequences,  39309 rows
additional data   : 36 sequences,  60098 rows
combined          : 59 sequences,  99407 rows


### Models we tried

The method is shared across motors, so instead of separate per-motor models we summarise
the approaches we tried and their Kaggle scores. The progression below is what took us
from about 0.10 to first place.

| Approach | Idea | Kaggle private / public | Verdict |
|---|---|---|---|
| Naive classifier | threshold / tree on raw signals | 0.10 to 0.35 | predicts almost nothing useful |
| SMOTE + RandomForest | balanced supervised classifier on engineered features | ~0.62 | collapses on blind test (covariate shift) |
| Learned regression twin | predict temperature, threshold absolute residual | ~0.68 | over-flags, destroys precision |
| Residual + hysteresis + synthetic augmentation | per-sequence robust residual, shape-aware hysteresis, generator-matched synthetic calibration | 0.843 / 0.752 | transfers; no train/test model |
| above + window 500 | larger rolling baseline | 0.853 / 0.790 | best public score |
| above + window 600 (final) | larger still, validated on real held-out faults | 0.863 / 0.762 | best private, 1st place |

The unsupervised residual detector is the qualitative jump; augmentation made the local
metric trustworthy; the window size was the final tuning lever.

In [2]:
# Final calibrated detector configuration.
FINAL = {m: dict(window=600, k_high=4.0, k_low=1.0, min_rise=0.5, min_run=3, gap=3)
         for m in range(1, 6)}                 # motors 1-5: shared knobs
FINAL[6] = dict(window=300, k_high=4.0, k_low=1.0, min_rise=0.5, min_run=3, gap=3)  # motor 6
pd.DataFrame(FINAL).T.rename_axis('motor')

,window,k_high,k_low,min_rise,min_run,gap
motor,,,,,,
1,600.0,4.0,1.0,0.5,3.0,3.0
2,600.0,4.0,1.0,0.5,3.0,3.0
3,600.0,4.0,1.0,0.5,3.0,3.0
4,600.0,4.0,1.0,0.5,3.0,3.0
5,600.0,4.0,1.0,0.5,3.0,3.0
6,300.0,4.0,1.0,0.5,3.0,3.0


### Per-motor performance

Reporting performance for this challenge needs care. The labelled-training F1 is not a
reliable indicator here: motors 1 to 5 have a single, subtle, real fault sequence (and
the additional groups are different robots, harder still), so a detector tuned for the
real test can look poor on those few labelled faults.

The metric that did track the leaderboard is a held-out synthetic generator-matched F1:
inject faults with the competition's own routine (peaks across its full randi[2,50]
range, on held-out seeds) and score per motor. Its mean (about 0.85) is close to the
0.863 private leaderboard score.

In [3]:
from sklearn.metrics import f1_score

def synthetic_f1(df, motor, knobs, seeds=(50, 51, 52)):
    '''Per-motor F1 on faults injected by the competition's own generator into
    held-out normal sequences (peaks over the full randi[2,50] range).'''
    units = (sw.normal_units(df, [motor])
             + sw.synth(df, [motor], seeds, peak_range=(2, 50),
                        span_frac=(0.03, 0.25), n_per_seq=3))
    res = [det.prep_residual(u['temp'], knobs['window']) for u in units]
    truth = np.concatenate([u['truth'] for u in units])
    preds = np.concatenate([
        det.detect_from_scored(r[0], r[1], knobs['k_high'], knobs['k_low'],
                               knobs['min_rise'], knobs['min_run'], knobs['gap'])
        for r in res])
    return f1_score(truth, preds, zero_division=0)

perf = pd.DataFrame([
    {'motor': m, 'window': FINAL[m]['window'],
     'synthetic CV-F1 (Kaggle proxy)': round(synthetic_f1(df_all, m, FINAL[m]), 3)}
    for m in range(1, 7)])
print(perf.to_string(index=False))
print(f"\nmean per-motor synthetic CV-F1 : {perf['synthetic CV-F1 (Kaggle proxy)'].mean():.3f}"
      f"   (vs Kaggle private 0.863)")

 motor  window  synthetic CV-F1 (Kaggle proxy)
     1     600                           0.848
     2     600                           0.925
     3     600                           0.849
     4     600                           0.865
     5     600                           0.869
     6     300                           0.749

mean per-motor synthetic CV-F1 : 0.851   (vs Kaggle private 0.863)


### Prepare final submission

We apply the final detector to the test set and write the Kaggle submission. This
reproduces submission_w600.csv, our first-place entry.

In [4]:
df_test = rp.load(rp.TEST_DIR)
sub = pd.read_csv(rp.SAMPLE_SUB)
for m in range(1, 7):
    col = f'data_motor_{m}_label'
    preds = det.detect_grouped(df_test, f'data_motor_{m}_temperature', FINAL[m])
    for tc in df_test['test_condition'].unique():
        tmask = (df_test['test_condition'] == tc).to_numpy()
        sidx = sub.index[sub['test_condition'] == tc]
        p = preds[tmask]; n = min(len(sidx), len(p))
        sub.loc[sidx[:n], col] = p[:n].astype(int)
    sub[col] = sub[col].replace(-1, 0).astype(int)

sub.to_csv(rp.OUT_SUB, index=False)
print('rows:', len(sub), '| remaining -1 placeholders:', int((sub.filter(like='_label') == -1).sum().sum()))
print('faults flagged per motor (% of rows):',
      {m: round(float((sub[f'data_motor_{m}_label'] == 1).mean() * 100), 2) for m in range(1, 7)})
sub.head()

rows: 14157 | remaining -1 placeholders: 0
faults flagged per motor (% of rows): {1: 1.8, 2: 1.33, 3: 3.99, 4: 3.5, 5: 1.53, 6: 2.84}


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,0,0,0,0,0,20240527_094865
1,1,0,0,0,0,0,0,20240527_094865
2,2,0,0,0,0,0,0,20240527_094865
3,3,0,0,0,0,0,0,20240527_094865
4,4,0,0,0,0,0,0,20240527_094865


## Discussions and Conclusions

### Final scores
Our final submission (submission_w600.csv) reached private 0.86323 and public 0.76204,
taking 1st place. The key submissions:

| Submission | Private | Public |
|---|---|---|
| residual + augmentation (synthetic) | 0.84266 | 0.75190 |
| window 500 | 0.85329 | 0.78992 |
| window 600 (final) | 0.86323 | 0.76204 |

The window-500 entry led the public board (0.790) while window-600 won the private one
(0.863). The public split is a partial, noisier view of the test set, so the entry that
wins is the one validated on a distribution-faithful held-out set rather than the one
with the best public score.

### Why the final model works
- No train/test model dependency. Detection is unsupervised per sequence and
  scale-invariant (per-sequence median and MAD), so it does not suffer the covariate
  shift that sank the classifier and the learned twin.
- Shape-awareness. Hysteresis matches the fault's asymmetric triangle, so it catches
  subtle ramp tails (especially on motors 3 and 6) without over-flagging.
- A trustworthy local metric. Because the test labels share the injection generator,
  our synthetic held-out F1 (about 0.85) tracked the leaderboard, so we could tune
  offline. This was the most important choice we made.

### Strengths and limitations
- Strength: robustness and transfer. The detector needs no positive examples to run
  (labels only calibrate a few knobs), which is why it survives the severe class
  imbalance.
- Limitation: the method is specific to the synthetic fault shape (a monotone
  temperature rise) and would need rethinking for real, noisier faults. Motor 6
  (gripper) stays the weakest (about 0.75), with heterogeneous, weaker faults.
- Caveat: the labelled-training F1 is an unreliable guide here, so we relied on the
  synthetic proxy and the real held-out additional data instead.

### Room for improvement
- A template or matched filter correlating the residual against the explicit rise/cool
  triangle (a bank of durations) would likely lift motors 3 and 6.
- A precision-weighted objective (F-beta, beta below 1) could use the scoring rule,
  which awards F1=1.0 to a motor whose true and predicted columns are both empty, so
  false positives on fault-free test motors are costly.
- Load-aware gating (a temperature rise not explained by voltage or position) could
  further separate genuine faults from legitimate warming.
